In [7]:
# Install required library
!pip install nba_api

In [8]:
# Fetch NBA player stats for seasons 2010-2024 using nba_api
from nba_api.stats.endpoints import leaguedashplayerstats
import pandas as pd
import time

seasons = [
    "2010-11", "2011-12", "2012-13", "2013-14", "2014-15",
    "2015-16", "2016-17", "2017-18", "2018-19", "2019-20",
    "2020-21", "2021-22", "2022-23", "2023-24"
]

all_data = []
for season in seasons:
    print(f"{season} fetching...")
    stats = leaguedashplayerstats.LeagueDashPlayerStats(
        season=season,
        per_mode_detailed="PerGame"
    )
    df = stats.get_data_frames()[0]
    df["SEASON"] = season
    all_data.append(df)
    time.sleep(0.5)

nba_df = pd.concat(all_data, ignore_index=True)
print(f"Total rows: {len(nba_df)}")

2010-11 fetching...
2011-12 fetching...
2012-13 fetching...
2013-14 fetching...
2014-15 fetching...
2015-16 fetching...
2016-17 fetching...
2017-18 fetching...
2018-19 fetching...
2019-20 fetching...
2020-21 fetching...
2021-22 fetching...
2022-23 fetching...
2023-24 fetching...
Total rows: 7190


In [9]:
# Select relevant columns, compute True Shooting %, filter for 1000+ minutes
columns_needed = [
    "PLAYER_ID", "PLAYER_NAME", "TEAM_ABBREVIATION", "AGE", "GP",
    "MIN", "PTS", "AST", "TOV", "REB",
    "STL", "BLK", "FG3_PCT", "FT_PCT",
    "FGA", "FTA", "FGM", "PLUS_MINUS", "SEASON"
]

nba_clean = nba_df[columns_needed].copy()
nba_clean["TS_PCT"] = nba_clean["PTS"] / (2 * (nba_clean["FGA"] + 0.44 * nba_clean["FTA"]))
nba_clean = nba_clean[nba_clean["MIN"] * nba_clean["GP"] >= 1000]
print(f"Filtered rows: {len(nba_clean)}")

Filtered rows: 3683


In [10]:
from google.colab import files
uploaded = files.upload()

Saving archive.zip to archive (1).zip


In [11]:
# Upload Kaggle dataset containing position and advanced stats
# Merge nba_api data with Kaggle dataset on player name and season
kaggle_df = pd.read_csv("archive.zip")

kaggle_positions = kaggle_df[["normalized_name", "season", "Pos.x", "TS.", "PER", "BPM", "VORP", "WS"]].copy()
kaggle_positions = kaggle_positions.rename(columns={
    "normalized_name": "PLAYER_NAME",
    "season": "SEASON",
    "Pos.x": "POSITION",
    "TS.": "TS_PCT_KAG"
})

nba_final = nba_clean.merge(kaggle_positions, on=["PLAYER_NAME", "SEASON"], how="inner")
print(f"Merged rows: {len(nba_final)}")
print(nba_final["POSITION"].value_counts())

Merged rows: 3570
POSITION
SG    829
PG    738
PF    725
SF    668
C     610
Name: count, dtype: int64


In [12]:
# Save final dataset as CSV
nba_final.to_csv("nba_final.csv", index=False)
from google.colab import files
files.download("nba_final.csv")
print("Done!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done!
